In [37]:
%pip install pandas scikit-learn imbalanced-learn


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [38]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from imblearn.datasets import make_imbalance

In [ ]:
print("Baixando o dataset do OpenML...")
dataset = fetch_openml(name='spambase', version=1, as_frame=True, parser='liac-arff')
X_real = dataset.data

y_real = LabelEncoder().fit_transform(dataset.target)

bincount = np.bincount(y_real)
classe_maj = np.argmax(bincount)
classe_min = np.argmin(bincount)
n_maj_original = bincount[classe_maj]

print(f"Dataset carregado! Distribuição original: {bincount}")

Baixando o dataset do OpenML...
Dataset carregado! Distribuição original: [2788 1813]


In [ ]:
classifiers = {
    'KNN': KNeighborsClassifier(),
    'RandomForest': RandomForestClassifier(random_state=42),
    'SVC': SVC(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42) 
}

In [ ]:
samplers = {
    'Baseline (Sem tratamento)': None,
    'RUS (Undersampling)': RandomUnderSampler(random_state=42),
    'SMOTE (Oversampling Clássico)': SMOTE(random_state=42),
    'ADASYN (Oversampling Avançado)': ADASYN(random_state=42),
    'SMOTETomek (Híbrido)': SMOTETomek(random_state=42)
}

In [ ]:
scoring = ['accuracy', 'precision', 'recall', 'f1', 'average_precision'] # average_precision = PR-AUC

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
f1_numeric_for_sorting = []

In [ ]:
niveis_minoritaria = [0.10, 0.05, 0.02, 0.01]

for proporcao in niveis_minoritaria:
    
    n_min_desejado = int(n_maj_original * (proporcao / (1 - proporcao)))
    
    if n_min_desejado > bincount[classe_min]:
         print(f"\nAviso: Não há exemplos suficientes na classe minoritária para {proporcao*100}%.")
         continue
         
    estrategia = {classe_maj: n_maj_original, classe_min: n_min_desejado}
    X, y = make_imbalance(X_real, y_real, sampling_strategy=estrategia, random_state=42)
    
    print(f"Testando nível {int(proporcao*100)}% (Majoritária: {n_maj_original}, Minoritária: {n_min_desejado})")
    
    for clf_name, clf in classifiers.items():
        for sampler_name, sampler in samplers.items():
            
            pipeline = Pipeline([('classifier', clf)]) if sampler is None else Pipeline([('sampler', sampler), ('classifier', clf)])
                
            scores = cross_validate(pipeline, X, y, scoring=scoring, cv=cv, n_jobs=-1)
            
            results.append({
                'Minoria %': f"{int(proporcao*100)}%",
                'Classificador': clf_name,
                'Estratégia': sampler_name,
                #'Acurácia': f"{np.mean(scores['test_accuracy']):.4f} ± {np.std(scores['test_accuracy']):.4f}",
                #'Precisão': f"{np.mean(scores['test_precision']):.4f} ± {np.std(scores['test_precision']):.4f}",
                #'Revocação': f"{np.mean(scores['test_recall']):.4f} ± {np.std(scores['test_recall']):.4f}",
                'F1-Score': f"{np.mean(scores['test_f1']):.4f} ± {np.std(scores['test_f1']):.4f}",
                'PR-AUC': f"{np.mean(scores['test_average_precision']):.4f} ± {np.std(scores['test_average_precision']):.4f}"
            })
            f1_numeric_for_sorting.append(np.mean(scores['test_f1']))

Testando nível 10% (Majoritária: 2788, Minoritária: 309)
Testando nível 5% (Majoritária: 2788, Minoritária: 146)
Testando nível 2% (Majoritária: 2788, Minoritária: 56)


/run/media/system/BazziteSSD/Artigos/ArtigoMD/.venv/lib64/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/run/media/system/BazziteSSD/Artigos/ArtigoMD/.venv/lib64/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/run/media/system/BazziteSSD/Artigos/ArtigoMD/.venv/lib64/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Testando nível 1% (Majoritária: 2788, Minoritária: 28)


/run/media/system/BazziteSSD/Artigos/ArtigoMD/.venv/lib64/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/run/media/system/BazziteSSD/Artigos/ArtigoMD/.venv/lib64/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/run/media/system/BazziteSSD/Artigos/ArtigoMD/.venv/lib64/python3.15/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

In [ ]:
df_results = pd.DataFrame(results)
df_results['F1_Num'] = f1_numeric_for_sorting

df_results = df_results.sort_values(by=['Classificador' ,'Minoria %' ,'F1_Num'], ascending=[True, False, False]).drop(columns=['F1_Num'])

display(df_results)
df_results.to_csv('dados_teste_md.csv')

for nivel in df_results['Minoria %'].unique():
    df_filtrado = df_results[df_results['Minoria %'] == nivel]
    
    nivel_formatado = nivel.replace('%', 'pct')
    nome_arquivo = f"dados_teste_md_{nivel_formatado}.csv"
    
    df_filtrado.to_csv(nome_arquivo, index=False)
    print(f"Arquivo gerado: {nome_arquivo}")

,Minoria %,Classificador,Estratégia,F1-Score,PR-AUC
35,5%,Decision Tree,Baseline (Sem tratamento),0.5920 ± 0.1134,0.3912 ± 0.1204
38,5%,Decision Tree,ADASYN (Oversampling Avançado),0.5701 ± 0.0974,0.3690 ± 0.0918
37,5%,Decision Tree,SMOTE (Oversampling Clássico),0.5466 ± 0.1291,0.3516 ± 0.1119
39,5%,Decision Tree,SMOTETomek (Híbrido),0.5457 ± 0.1091,0.3486 ± 0.0924
36,5%,Decision Tree,RUS (Undersampling),0.2965 ± 0.0877,0.1627 ± 0.0520
...,...,...,...,...,...
71,1%,SVC,RUS (Undersampling),0.0518 ± 0.0270,0.0448 ± 0.0328
74,1%,SVC,SMOTETomek (Híbrido),0.0317 ± 0.0086,0.0538 ± 0.0208
72,1%,SVC,SMOTE (Oversampling Clássico),0.0311 ± 0.0090,0.0705 ± 0.0540
73,1%,SVC,ADASYN (Oversampling Avançado),0.0299 ± 0.0091,0.0718 ± 0.0531


Arquivo gerado: dados_teste_md_5pct.csv
Arquivo gerado: dados_teste_md_2pct.csv
Arquivo gerado: dados_teste_md_10pct.csv
Arquivo gerado: dados_teste_md_1pct.csv
